# 05 - Event Study & Analysis Dataset

Joins storm events, ZHVI panel, and NRI data to produce the final analysis dataset.

**Inputs:**
- `../data/processed/storm_events.pkl` — county-month storm observations
- `../data/processed/zillow_panel.pkl` — affected county-month ZHVI vs regional baseline
- `../data/processed/nri_panel.pkl` — NRI scores by county-year
- Zillow raw CSV — for pre-storm ZHVI lookups

**Output:** `../data/processed/analysis_dataset.pkl` — one row per county-month storm event
with complete 12-month post-storm windows

**Measures computed:**
- `auc` — cumulative sum of monthly ZHVI deviation from regional baseline over T+1 to T+12
- `auc_variance` — standard deviation of those 12 monthly deviations
- `pre_trend_annual` — annualized pre-storm drift rate (avg monthly deviation T-6 to T-1) x 12

**Window requirements:**
- 12 months post-storm ZHVI required (incomplete windows dropped)
- 6 months pre-storm ZHVI required for pre_trend_annual
- Latest ZHVI data: February 2026 → latest eligible storm month: February 2025

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

## Load Processed Data

In [2]:
storms   = pd.read_pickle('../data/processed/storm_events.pkl')
zillow   = pd.read_pickle('../data/processed/zillow_panel.pkl')
nri      = pd.read_pickle('../data/processed/nri_panel.pkl')

print(f'Storm events:  {storms.shape}')
print(f'Zillow panel:  {zillow.shape}')
print(f'NRI panel:     {nri.shape}')

Storm events:  (48978, 7)
Zillow panel:  (47384, 10)
NRI panel:     (18870, 12)


## Load Full Zillow ZHVI for Pre-Storm Window

The processed zillow_panel only covers 2020-2025. Pre-storm windows
for early 2020 events require ZHVI from mid-2019, so we load the
full raw dataset and build a lookup.

In [3]:
# Load full county ZHVI wide format
county_raw = pd.read_csv('../data/raw/zillow_county_zhvi.csv')

META_COLS = ['RegionID', 'SizeRank', 'RegionName', 'RegionType',
             'StateName', 'State', 'Metro', 'StateCodeFIPS', 'MunicipalCodeFIPS',
             'latitude', 'longitude']

date_cols = [c for c in county_raw.columns if c not in META_COLS]

# Build long format for all dates
zhvi_long = county_raw.melt(
    id_vars=['StateCodeFIPS', 'MunicipalCodeFIPS'],
    value_vars=date_cols,
    var_name='date',
    value_name='zhvi'
)
zhvi_long['date'] = pd.to_datetime(zhvi_long['date'])
zhvi_long['state_fips']  = zhvi_long['StateCodeFIPS'].astype(str).str.zfill(2)
zhvi_long['county_fips'] = zhvi_long['MunicipalCodeFIPS'].astype(str).str.zfill(3)
zhvi_long['stcofips']    = zhvi_long['state_fips'] + zhvi_long['county_fips']
zhvi_long['year']        = zhvi_long['date'].dt.year
zhvi_long['month']       = zhvi_long['date'].dt.month
zhvi_long = zhvi_long.dropna(subset=['zhvi'])

# Index by stcofips + year + month for fast lookup
zhvi_idx = zhvi_long.set_index(['stcofips', 'year', 'month'])['zhvi']

print(f'ZHVI lookup: {len(zhvi_idx):,} county-month records')
print(f'Date range: {zhvi_long["date"].min().date()} to {zhvi_long["date"].max().date()}')

ZHVI lookup: 690,676 county-month records
Date range: 2000-01-31 to 2026-02-28


## Build Regional Baseline Lookup

We need baseline ZHVI for pre-storm months too, not just post-storm.
Recompute baseline from the full zillow panel using the same logic as notebook 04.

In [4]:
# Regional baseline is already in zillow_panel for affected county-months
# Build a region x year x month baseline lookup from zillow_panel
# Load pre-computed baseline including fallback months
baseline_df = pd.read_pickle('../data/processed/baseline_lookup.pkl')
baseline_lookup = baseline_df.set_index(['climate_region', 'year', 'month'])['baseline_zhvi']

# Also need climate_region per county for pre-storm baseline lookups
county_region = (
    zillow[['stcofips', 'climate_region']]
    .drop_duplicates()
    .set_index('stcofips')['climate_region']
)

print(f'Baseline lookup: {len(baseline_lookup):,} region-month records')
print(f'Coverage by region:')
print(baseline_df.groupby('climate_region').size().reset_index(name='months'))
print(f'County-region mapping: {len(county_region):,} counties')

Baseline lookup: 648 region-month records
Coverage by region:
       climate_region  months
0             Central      72
1  East North Central      72
2           Northeast      72
3           Northwest      72
4               South      72
5           Southeast      72
6           Southwest      72
7                West      72
8  West North Central      72
County-region mapping: 3,051 counties


## Filter to Eligible Storm Events

Drop events without a complete 12-month post-storm window.
Latest ZHVI is February 2026, so storm month must be ≤ February 2025.

In [5]:
# Convert year/month to period for cutoff comparison
storms['period'] = pd.to_datetime(
    storms[['year', 'month']].assign(day=1)
)

LATEST_ZHVI     = pd.Timestamp('2026-02-28')
CUTOFF          = LATEST_ZHVI - pd.DateOffset(months=12)

eligible = storms[storms['period'] <= CUTOFF].copy()

print(f'Total storm events:    {len(storms):,}')
print(f'Eligible (12mo window): {len(eligible):,}')
print(f'Dropped (incomplete):  {len(storms) - len(eligible):,}')
print(f'Cutoff storm month:    {CUTOFF.strftime("%Y-%m")}')

Total storm events:    48,978
Eligible (12mo window): 41,639
Dropped (incomplete):  7,339
Cutoff storm month:    2025-02


## Compute AUC, Variance, and Pre-Trend per Event

In [6]:
def compute_event_metrics(row, zhvi_idx, baseline_lookup, county_region):
    """
    For a single county-month storm event, compute:
    - auc: cumulative sum of ZHVI deviation from baseline over T+1 to T+12
    - auc_variance: std dev of those 12 monthly deviations
    - pre_trend_annual: annualized pre-storm drift (avg deviation T-6 to T-1) x 12

    ZHVI deviation = county ZHVI - regional baseline ZHVI
    All values in absolute dollar terms (not normalized to 100).
    Returns None if any required window months are missing.
    """
    fips   = row['stcofips']
    year   = row['year']
    month  = row['month']
    region = county_region.get(fips)

    if region is None:
        return None

    storm_date = pd.Timestamp(year=year, month=month, day=1)


    # Get T=0 values for indexing
    county_zhvi_t0   = zhvi_idx.get((fips, year, month))
    baseline_zhvi_t0 = baseline_lookup.get((region, year, month))

    if county_zhvi_t0 is None or baseline_zhvi_t0 is None:
        return None
    if pd.isna(county_zhvi_t0) or pd.isna(baseline_zhvi_t0):
        return None


    # --- Post-storm window: T+1 to T+12 ---
    post_deviations = []
    for n in range(1, 13):
        future = storm_date + pd.DateOffset(months=n)
        fy, fm = future.year, future.month
        county_zhvi   = zhvi_idx.get((fips, fy, fm))
        baseline_zhvi = baseline_lookup.get((region, fy, fm))
        if county_zhvi is None or baseline_zhvi is None or pd.isna(county_zhvi) or pd.isna(baseline_zhvi):
            return None
        county_idx   = (county_zhvi   / county_zhvi_t0)   * 100
        baseline_idx = (baseline_zhvi / baseline_zhvi_t0) * 100
        post_deviations.append(county_idx - baseline_idx)


    # --- Pre-storm window: T-6 to T-1 ---
    pre_deviations = []
    for n in range(1, 7):
        past = storm_date - pd.DateOffset(months=n)
        py, pm = past.year, past.month
        county_zhvi   = zhvi_idx.get((fips, py, pm))
        baseline_zhvi = baseline_lookup.get((region, py, pm))
        if county_zhvi is None or baseline_zhvi is None or pd.isna(county_zhvi) or pd.isna(baseline_zhvi):
            pre_deviations.append(np.nan)
        else:
            county_idx   = (county_zhvi   / county_zhvi_t0)   * 100
            baseline_idx = (baseline_zhvi / baseline_zhvi_t0) * 100
            pre_deviations.append(county_idx - baseline_idx)

    post_arr = np.array(post_deviations)
    pre_arr  = np.array(pre_deviations)

    auc             = float(np.sum(post_arr))
    auc_variance    = float(np.std(post_arr, ddof=1))
    pre_trend_monthly = float(np.nanmean(pre_arr)) if not np.all(np.isnan(pre_arr)) else np.nan
    pre_trend_annual  = pre_trend_monthly * 12 if not np.isnan(pre_trend_monthly) else np.nan

    return {
        'auc':              auc,
        'auc_variance':     auc_variance,
        'pre_trend_annual': pre_trend_annual,
        'post_deviations':  post_arr.tolist(),
        'pre_deviations':   pre_arr.tolist(),
    }

print('compute_event_metrics defined')

compute_event_metrics defined


In [7]:
# Apply to all eligible events
print(f'Computing metrics for {len(eligible):,} events...')

results = []
dropped = 0
for _, row in eligible.iterrows():
    metrics = compute_event_metrics(row, zhvi_idx, baseline_lookup, county_region)
    if metrics is None:
        dropped += 1
    else:
        results.append({**row.to_dict(), **metrics})

metrics_df = pd.DataFrame(results)
print(f'Events with complete windows: {len(metrics_df):,}')
print(f'Dropped (missing ZHVI in window): {dropped:,}')

Computing metrics for 41,639 events...
Events with complete windows: 39,838
Dropped (missing ZHVI in window): 1,801


## Join Storm Severity and NRI

In [8]:
# Join climate region
metrics_df['climate_region'] = metrics_df['stcofips'].map(county_region)

# Join NRI — match on stcofips + storm year
nri_cols = ['stcofips', 'storm_year', 'nri_vintage', 'resl_score', 'risk_value', 'eal_valt', 'sovi_score']
metrics_df = metrics_df.merge(
    nri[nri_cols].rename(columns={'storm_year': 'year'}),
    on=['stcofips', 'year'],
    how='left'
)

print(f'Shape after joins: {metrics_df.shape}')
print(f'Missing NRI: {metrics_df["resl_score"].isnull().sum()}')

Shape after joins: (39838, 19)
Missing NRI: 96


In [9]:
missing_nri = metrics_df['resl_score'].isnull()
print(f'Dropping {missing_nri.sum()} rows with missing NRI (CT county restructuring edge case)')
metrics_df = metrics_df[~missing_nri].copy()

assert metrics_df.duplicated(['stcofips', 'year', 'month']).sum() == 0
assert metrics_df['auc'].notnull().all()
assert metrics_df['auc_variance'].notnull().all()
assert metrics_df['resl_score'].isnull().sum() == 0
assert metrics_df['climate_region'].isnull().sum() == 0
print('All assertions passed')

Dropping 96 rows with missing NRI (CT county restructuring edge case)
All assertions passed


## Validate

In [10]:
assert metrics_df.duplicated(['stcofips', 'year', 'month']).sum() == 0, 'Duplicate county-month rows'
assert metrics_df['auc'].notnull().all(), 'Null AUC values'
assert metrics_df['auc_variance'].notnull().all(), 'Null AUC variance values'
assert metrics_df['resl_score'].isnull().sum() == 0, 'Missing NRI scores'
assert metrics_df['climate_region'].isnull().sum() == 0, 'Missing climate region'

print('All assertions passed')
print(f'\nFinal dataset summary:')
print(metrics_df[['auc', 'auc_variance', 'pre_trend_annual', 'resl_score', 'log_damage', 'event_count']].describe().round(2))

All assertions passed

Final dataset summary:
            auc  auc_variance  pre_trend_annual  resl_score  log_damage  \
count  39742.00      39742.00          39131.00    39742.00    39742.00   
mean      -0.26          3.85             -4.72       53.39        4.22   
std       70.02          2.17             65.91       16.72        5.15   
min     -326.09          0.78           -313.69        0.03        0.00   
25%      -39.55          2.39            -40.50       51.45        0.00   
50%        0.67          3.28             -3.48       54.77        0.00   
75%       38.41          4.52             29.72       57.64        9.21   
max      366.29         18.38            389.54      100.00       22.41   

       event_count  
count     39742.00  
mean          5.48  
std           8.68  
min           1.00  
25%           1.00  
50%           3.00  
75%           6.00  
max         330.00  


## Export

In [11]:
monthly_rows = []
for _, row in metrics_df.iterrows():
    for t, dev in enumerate(row['post_deviations'], start=1):
        monthly_rows.append({
            'stcofips': row['stcofips'],
            'year':     row['year'],
            'month':    row['month'],
            'month_t':  t,
            'deviation': dev,
        })
        
for _, row in metrics_df.iterrows():
    monthly_rows.append({
        'stcofips': row['stcofips'],
        'year':     row['year'],
        'month':    row['month'],
        'month_t':  0,
        'deviation': 0.0,
    })

monthly_deviations = pd.DataFrame(monthly_rows)

# Unpack pre-storm deviations into long format (T-6 to T-1)
pre_rows = []
for _, row in metrics_df.iterrows():
    for t, dev in enumerate(row['pre_deviations'], start=1):
        if not np.isnan(dev):
            pre_rows.append({
                'stcofips': row['stcofips'],
                'year':     row['year'],
                'month':    row['month'],
                'month_t':  -t,
                'deviation': dev,
            })

pre_deviations_df = pd.DataFrame(pre_rows)

# Combine pre and post into single file
all_deviations = pd.concat([pre_deviations_df, monthly_deviations], ignore_index=True)
all_deviations.to_csv('../data/processed/monthly_deviations.csv', index=False)
print(f'Exported monthly_deviations.csv: {all_deviations.shape}')

metrics_df = metrics_df.drop(columns=['post_deviations', 'pre_deviations'])
print(f'Dropped cols: post_deviations, pre_deviations')

Exported monthly_deviations.csv: (737735, 5)
Dropped cols: post_deviations, pre_deviations


In [12]:
output_cols = [
    # Identifiers
    'stcofips', 'year', 'month', 'climate_region',
    # Outcome measures
    'auc', 'auc_variance', 'pre_trend_annual',
    # Storm severity
    'event_count', 'total_damage', 'log_damage', 'total_duration_days',
    # NRI
    'resl_score', 'risk_value', 'eal_valt', 'sovi_score', 'nri_vintage',
]

out = metrics_df[output_cols].sort_values(['stcofips', 'year', 'month']).reset_index(drop=True)

out.to_pickle('../data/processed/analysis_dataset.pkl')

# Also export as CSV for R
out.to_csv('../data/processed/analysis_dataset.csv', index=False)

print(f'Exported analysis_dataset.pkl and analysis_dataset.csv')
print(f'Shape: {out.shape}')
print(f'Columns: {out.columns.tolist()}')

Exported analysis_dataset.pkl and analysis_dataset.csv
Shape: (39742, 16)
Columns: ['stcofips', 'year', 'month', 'climate_region', 'auc', 'auc_variance', 'pre_trend_annual', 'event_count', 'total_damage', 'log_damage', 'total_duration_days', 'resl_score', 'risk_value', 'eal_valt', 'sovi_score', 'nri_vintage']


In [13]:
import pandas as pd
zp = pd.read_pickle('../data/processed/zillow_panel.pkl')
print(f"Fallback pct: {(zp['baseline_n_counties'] == 0).mean():.1%}")
print(f"Fallback rows: {(zp['baseline_n_counties'] == 0).sum():,} of {len(zp):,}")

Fallback pct: 0.0%
Fallback rows: 0 of 47,384
